In [ ]:
!pip install pdfplumber pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 100.6 MB/s eta 0:00:00


In [ ]:
import pdfplumber, re, pandas as pd, os
from tqdm import tqdm

# 📂 Set your Drive folder path here
pdf_folder = "/content/drive/MyDrive/Waterfootprint"

# Define an empty list to collect data
records = []

# Loop through all PDF files in the folder
for file in tqdm(os.listdir(pdf_folder)):
    if file.lower().endswith(".pdf"):
        file_path = os.path.join(pdf_folder, file)

        # Try to guess Region and Crop from filename
        name_parts = file.replace(".pdf","").split()
        region = name_parts[0] if len(name_parts) > 0 else "Unknown"
        crop   = name_parts[1] if len(name_parts) > 1 else "Unknown"

        # Extract text using pdfplumber
        with pdfplumber.open(file_path) as pdf:
            full_text = ""
            for page in pdf.pages:
                txt = page.extract_text()
                if txt: full_text += txt + "\n"

        # Find numbers near keywords like "litre", "mm", "cm", "water"
        pattern = r"(\d{3,5})\s*(?:litres?|mm|cm|L|l)"
        matches = re.findall(pattern, full_text)
        if matches:
            avg_value = sum(map(int, matches)) / len(matches)
        else:
            avg_value = None

        # Estimate qualitative level based on value (dummy logic)
        if avg_value:
            if avg_value > 3000:
                category = "High"
            elif avg_value > 1500:
                category = "Medium"
            else:
                category = "Low"
        else:
            category = "Unknown"

        records.append({
            "Region": region,
            "Crop": crop,
            "Avg_Water_Requirement(L_per_kg)": avg_value,
            "Category": category,
            "Source_File": file
        })

# Convert to DataFrame
df = pd.DataFrame(records)
df.to_csv("dummy_water_footprint_dataset.csv", index=False)
df.head()


100%|██████████| 6/6 [00:09<00:00,  1.59s/it]


,Region,Crop,Avg_Water_Requirement(L_per_kg),Category,Source_File
0,kolhapur_sugarcane_removed,Unknown,None,Unknown,kolhapur_sugarcane_removed.pdf
1,pune-wheat_removed,Unknown,None,Unknown,pune-wheat_removed.pdf
2,pune-gram_removed,Unknown,None,Unknown,pune-gram_removed.pdf
3,kolhapur-wheat_removed,Unknown,None,Unknown,kolhapur-wheat_removed.pdf
4,pune-cotton_removed,Unknown,None,Unknown,pune-cotton_removed.pdf


In [1]:
!pip install pdf2image pytesseract pdfplumber pandas tqdm pillow

import os, re, pandas as pd
from tqdm import tqdm
from pdf2image import convert_from_path
import pytesseract
import pdfplumber

# 📂 Your Drive folder path
pdf_folder = "/content/drive/MyDrive/Waterfootprint"

records = []

def extract_text(path):
    """Extract text with pdfplumber, fallback to OCR if text is empty."""
    text = ""
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"
    except:
        pass

    if len(text.strip()) < 50:
        print(f"🟡 Using OCR for: {os.path.basename(path)}")
        images = convert_from_path(path)
        for img in images:
            text += pytesseract.image_to_string(img)
    return text


for file in tqdm(os.listdir(pdf_folder)):
    if not file.lower().endswith(".pdf"):
        continue

    file_path = os.path.join(pdf_folder, file)

    # Fix: split underscores to detect region and crop
    parts = file.replace(".pdf", "").replace("_", " ").split()
    region = parts[0].capitalize() if len(parts) > 0 else "Unknown"
    crop = parts[1].capitalize() if len(parts) > 1 else "Unknown"

    # Extract text (OCR if needed)
    full_text = extract_text(file_path)

    # Improved regex — catches integers, decimals, and unit variations
    pattern = r"(\d+[.,]?\d*)\s*(?:litres?|mm|cm|L|l|mm/day|mm per day|mm\/day)?"
    matches = re.findall(pattern, full_text)

    if matches:
        # Convert all numbers to float safely
        numeric_values = [float(x.replace(',', '')) for x in matches if x.strip()]
        avg_value = sum(numeric_values) / len(numeric_values)
    else:
        avg_value = None

    # Qualitative category (dummy classification)
    if avg_value:
        if avg_value > 3000:
            category = "High"
        elif avg_value > 1500:
            category = "Medium"
        else:
            category = "Low"
    else:
        category = "Unknown"

    records.append({
        "Region": region,
        "Crop": crop,
        "Avg_Water_Requirement(L_per_kg)": avg_value,
        "Category": category,
        "Source_File": file
    })

# Create DataFrame and save
df = pd.DataFrame(records)
df.to_csv("dummy_water_footprint_dataset.csv", index=False)
df.head()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 73.4 MB/s eta 0:00:00


100%|██████████| 48/48 [00:33<00:00,  1.42it/s]


,Region,Crop,Avg_Water_Requirement(L_per_kg),Category,Source_File
0,Kolhapur-wheat,Removed,7744.511628,High,kolhapur-wheat_removed.pdf
1,Kolapur-cotton,Removed,10171.523077,High,kolapur-cotton_removed.pdf
2,Pune-wheat,Removed,14678.311111,High,pune-wheat_removed.pdf
3,Pune-gram,Removed,16514.200000,High,pune-gram_removed.pdf
4,Pune-cotton,Removed,4767.784173,High,pune-cotton_removed.pdf
